# 03b. 蔵元の住所と水道水データの結合

蔵元ごとの**水の硬度**を求め、口コミに付与します（入力: 02, 03の出力 → 出力: `reviews_with_water_hardness.csv`）。

```
maker_address_master.csv ─┐
                          ├─▶ 住所に含まれる市区町村で照合 ─▶ maker_water_quality_master.csv ────┐
national_water_hardness_  ─┘   （硬度・pHは市区町村ごとの中央値）                                  ├─▶ reviews_with_water_hardness.csv
master.csv                                                         saketime_analyzed.csv ───────┘     （蔵元名で結合）
```

- 水道水データは浄水場ごとに複数あるため、**市区町村ごとの中央値**にまとめます。
- 蔵元の住所に市区町村名が含まれる最初のデータを採用します（同じ都道府県内のみ検索）。
- 蔵元名は**都道府県と組み合わせて**照合します（別の県の同名の酒造場を取り違えないため）。

## 設定

In [ ]:
# --- 実行環境の設定 ---------------------------------------------------------
# Colab: Google Driveをマウントし、BASE_DIR 配下を読み書きします。
# ローカル: 環境変数 SAKE_PROJECT_DIR でデータ置き場を指定できます。
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/作成したプログラム等/日本酒口コミ分析'  # 自分のDriveのパスに変更
except ImportError:
    BASE_DIR = os.environ.get('SAKE_PROJECT_DIR', './sake_review_analysis')

CSV_DIR = f'{BASE_DIR}/口コミ_CSV'
os.makedirs(CSV_DIR, exist_ok=True)

# 入力
MAKER_ADDRESS_CSV = f'{CSV_DIR}/maker_address_master.csv'            # 02: 蔵元の住所
WATER_CSV         = f'{CSV_DIR}/national_water_hardness_master.csv'  # 02: 全国の水道水データ
REVIEWS_CSV       = f'{CSV_DIR}/saketime_analyzed.csv'               # 03: 項目分解済みの口コミ
# 出力
MAKER_WATER_CSV = f'{CSV_DIR}/maker_water_quality_master.csv'        # 蔵元 × 水質（06で使用）
OUTPUT_CSV      = f'{CSV_DIR}/reviews_with_water_hardness.csv'       # 口コミ + 水質（04で使用）


## 1. 関数の定義

In [ ]:
import numpy as np
import pandas as pd

# 蔵元住所マスタの都道府県はローマ字なので、水道水データ（日本語）と照合できるよう変換する
PREF_JA = {
    'hokkaido': '北海道', 'aomori': '青森県', 'iwate': '岩手県', 'miyagi': '宮城県',
    'akita': '秋田県', 'yamagata': '山形県', 'fukushima': '福島県', 'ibaraki': '茨城県',
    'tochigi': '栃木県', 'gunma': '群馬県', 'saitama': '埼玉県', 'chiba': '千葉県',
    'tokyo': '東京都', 'kanagawa': '神奈川県', 'niigata': '新潟県', 'toyama': '富山県',
    'ishikawa': '石川県', 'fukui': '福井県', 'yamanashi': '山梨県', 'nagano': '長野県',
    'gifu': '岐阜県', 'shizuoka': '静岡県', 'aichi': '愛知県', 'mie': '三重県',
    'shiga': '滋賀県', 'kyoto': '京都府', 'osaka': '大阪府', 'hyogo': '兵庫県',
    'nara': '奈良県', 'wakayama': '和歌山県', 'tottori': '鳥取県', 'shimane': '島根県',
    'okayama': '岡山県', 'hiroshima': '広島県', 'yamaguchi': '山口県', 'tokushima': '徳島県',
    'kagawa': '香川県', 'ehime': '愛媛県', 'kochi': '高知県', 'fukuoka': '福岡県',
    'saga': '佐賀県', 'nagasaki': '長崎県', 'kumamoto': '熊本県', 'oita': '大分県',
}
UNKNOWN_ADDRESS = {'不明', 'nan'}


def pref_key(name):
    """都道府県名を「県・府・都」なしの形にそろえる（'山形県' も '山形' も '山形' に。北海道はそのまま）"""
    name = str(name).strip()
    return name if name == '北海道' else name.rstrip('都道府県')


def summarize_water_by_city(path):
    """水道水データを「都道府県 × 市区町村」ごとの総硬度・pHの中央値にまとめる"""
    df = pd.read_csv(path)
    out = pd.DataFrame({
        'pref_ja': df['都道府県'],
        'city': df['所在地'].astype(str),
        'hardness': pd.to_numeric(df['総硬度'], errors='coerce'),
        'ph': pd.to_numeric(df['pH'], errors='coerce'),
    })
    return out.groupby(['pref_ja', 'city'], as_index=False)[['hardness', 'ph']].median()


def build_maker_water_master(df_maker, water_city):
    """蔵元ごとに、住所に市区町村名を含む最初の水質データを紐付ける（同じ都道府県内のみ）"""
    water_by_pref = {pref: g for pref, g in water_city.groupby('pref_ja')}
    no_water = water_city.iloc[0:0]

    rows = []
    for m in df_maker.itertuples():
        address = str(m.address)
        if address in UNKNOWN_ADDRESS:
            continue
        candidates = water_by_pref.get(PREF_JA.get(m.prefecture, ''), no_water)
        hit = next((w for w in candidates.itertuples() if w.city in address), None)
        rows.append({
            'prefecture': m.prefecture, 'pref_key': pref_key(PREF_JA.get(m.prefecture, '')),
            'maker': m.maker, 'address': address,
            'match_city': hit.city if hit else '未特定',
            'water_hardness': hit.hardness if hit else np.nan,
            'water_ph': hit.ph if hit else np.nan,
        })
    return pd.DataFrame(rows)


def attach_water_to_reviews(df_reviews, maker_water):
    """口コミに蔵元の水質を付与する（都道府県 + 蔵元名で結合。口コミの件数は変えない）"""
    keys = ['pref_key', 'maker']
    master = maker_water[keys + ['match_city', 'water_hardness', 'water_ph']].copy()

    # 同じ県に同名の蔵元が重複していると口コミが増えるため、1行にする（硬度ありを優先）
    master['has_hardness'] = master['water_hardness'].notna()
    master = (master.sort_values('has_hardness', ascending=False, kind='stable')
                    .drop_duplicates(keys).drop(columns='has_hardness'))

    reviews = df_reviews.assign(pref_key=df_reviews['prefecture'].map(pref_key))
    merged = reviews.merge(master, on=keys, how='left', validate='m:1').drop(columns='pref_key')
    assert len(merged) == len(df_reviews), '結合で口コミの件数が変わりました'
    return merged

## 2. 蔵元 × 水質マスタの作成

In [ ]:
df_maker = pd.read_csv(MAKER_ADDRESS_CSV)
water_city = summarize_water_by_city(WATER_CSV)
print(f'蔵元: {len(df_maker)}件 / 水道水データ（市区町村単位）: {len(water_city)}件')

maker_water = build_maker_water_master(df_maker, water_city)
maker_water.to_csv(MAKER_WATER_CSV, index=False, encoding='utf-8-sig')

n_match = (maker_water['match_city'] != '未特定').sum()
print(f'市区町村を特定できた蔵元: {n_match} / {len(df_maker)} ({n_match / len(df_maker):.1%})')
print(f'保存: {MAKER_WATER_CSV}')

## 3. 口コミへの付与

In [ ]:
df_reviews = pd.read_csv(REVIEWS_CSV)
print(f'口コミ: {len(df_reviews)}件')

dup = maker_water[maker_water.duplicated(['pref_key', 'maker'], keep=False)]
if len(dup):
    print('⚠️ 同じ県・同じ名前で複数行ある蔵元（1行にまとめます）:')
    print(dup[['prefecture', 'maker', 'address']].to_string(index=False))

df_final = attach_water_to_reviews(df_reviews, maker_water)
df_final.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

n_linked = df_final['water_hardness'].notna().sum()
print(f'硬度を紐付けできた口コミ: {n_linked} / {len(df_final)} ({n_linked / len(df_final):.1%})')
print(f'保存: {OUTPUT_CSV}')
df_final[['maker', 'match_city', 'water_hardness', 'water_ph']].head(3)